# Analisi dati per Raggi X

A grandi linee:
1. **rivelazioni dei raggi X con contatore Geiger**
    - conteggi (in $20 \text{s}$) al variare della tensione ($V_\text{EHT}= 30, 20 \text{kV}$)
    - conteggi (in $20 \text{s}$) al variare della corrente ($V_\text{EHT}= 30\text{kV}$)
2. **assorbimento dei raggi X nei materiali** (fissata la corrente in regime lineare)
    - conteggi (in $20 \text{s}$) al variare dello spessore dell'alluminio
    - conteggi (in $20 \text{s}$) al variare dello spessore del piombo
3. **misura dello spettro dei raggi X**
   - conteggi (in $20 \text{s}$) al variare dell'angolo di diffrazione ($V_\text{EHT}= 30\text{kV}$, e con $V_\text{EHT}= 20\text{kV}$ al prim'ordine) per verificare la legge di Bragg
4. **misura dello spettro dei raggi X con filtro di nichel** (ripeti punto precedente)
5. **determinazione della dimensione interatomica** (punto 3 al variare del cristallo)


### utils

In [ ]:
# stats things

def meanCalc(values):
    """ returns mean of values (w/ error) weighted by errors of: values = List[[value1, err1], [value2, err2], ...] """
    mean = 0
    mean_err = 0

    for v in values:
        mean_err += 1 / (v[1])**2      # 1 / sigma^2
        mean     += v[0] / (v[1])**2   # value / sigma^2

    # right now mean_err = sum of weights
    mean /= mean_err

    # this is the true error
    mean_err = 1 / (mean_err)**0.5

    return mean, mean_err

In [2]:
# plotter/fitter class

from ROOT import TCanvas, TGraphErrors, TLegend, gPad, TF1, kBlue, kRed
from array import array

class fitPlotter():
    """ utility class to plot graphs and do fits with ROOT """
    def __init__(self, name=""):
        self._canvas = TCanvas() 
        self._name   = name
        self._graphs  = []
        self._legends = []

    def addGraph(self, x, y, x_err=None, y_err=None, title="graph", fit_formula="pol1", color=kRed):
        """ adds graph and eventually performs fit (else pass fit_formula=None) """
        # using c-style arrays breaks out of bound errors, so we have to implement some sanity checks
        if len(x) != len(y):
            raise IndexError("lenght of provided data are not the same")
        elif (x_err) and len(x_err) != len(x):
            raise IndexError("err_x lenght does not match x")
        elif (y_err) and len(y_err) != len(y):
            raise IndexError("err_x lenght does not match x")
        
        n = len(x)
        ex = x_err if x_err is not None else [0.0] * n
        ey = y_err if y_err is not None else [0.0] * n

        x_arr, y_arr = array('d', x), array('d', y)
        ex_arr, ey_arr = array('d', ex), array('d', ey)

        graph = TGraphErrors(n, x_arr, y_arr, ex_arr, ey_arr)
        graph.SetTitle(title)
        graph.SetMarkerStyle(20)
        graph.SetMarkerColor(color)
        graph.SetLineColor(color)

        leg = TLegend(0.12, 0.75, 0.45, 0.88)
        leg.SetBorderSize(1)
        leg.SetFillColor(0)
        leg.AddEntry(graph, "data points", "ple")

        if fit_formula:
            print(f"\n--- fit Results for: {title} ---")
            fit_res = graph.Fit(fit_formula, "SQ")
            func = graph.GetFunction(fit_formula)
            func.SetLineColor(kBlue)
            
            # fit stats
            chi2 = func.GetChisquare()
            ndf = func.GetNDF()
            pvalue = func.GetProb()
            print(f"Function: {fit_formula}")
            print(f"Chi2/NDF: {chi2:.4f} / {ndf}")
            print(f"p-value:  {pvalue:.4f}\n")

            # fit parameters stuff
            params = []
            for i in range(func.GetNpar()):
                name = func.GetParName(i)
                val  = func.GetParameter(i)
                err  = func.GetParError(i)
                print(f"{name}: {val:.4f} +/- {err:.4f}")
                params.append([val,err])
            print("-" * 32)

            leg.AddEntry(func, "fit function", "l")

        # fuck the garbage collector
        self._graphs.append(graph)
        self._legends.append(leg)

        # returning (masked) fit results (if fit was performed)
        if fit_formula: 
            return params
        else:
            return None

    def drawCanvas(self, dimX=1000, dimY=500):
        """ draws entire canvas """
        nGraphs = len(self._graphs)
        if nGraphs < 1: return
        
        cols = 2
        rows = (nGraphs + 1) // 2 
        
        self._canvas = TCanvas(self._name, self._name, dimX, dimY * rows)
        self._canvas.Divide(cols, rows)

        for i in range(nGraphs):
            self._canvas.cd(i+1)
            gPad.SetGrid() 
            self._graphs[i].Draw("AP")
            self._legends[i].Draw()

        self._canvas.Draw()

    def saveCanvas(self, fileName="canvas.png"):
        """ saves canvas: has logic to modify name (but its not that useful) """
        if fileName == "canvas.png" and self._name:
            fileName = self._name + ".png"
        self._canvas.SaveAs(fileName)

In [4]:
import numpy as np
import scipy.stats as stats

def calcola_test_z(media_campione, media_popolazione, dev_std_popolazione, n, alfa=0.05, tipo_test='bilaterale'):
    
    # 1. Calcolo dell'errore standard
    errore_standard = dev_std_popolazione / np.sqrt(n)
    
    # 2. Calcolo della statistica Z
    Z = (media_campione - media_popolazione) / errore_standard
    print(f"Statistica Z calcolata: {Z:.4f}")
    
    # 3. Calcolo del P-value in base al tipo di test
    if tipo_test == 'bilaterale':
        # Moltiplico per 2 perché guardo entrambe le code
        p_value = 2 * (1 - stats.norm.cdf(abs(Z)))
    elif tipo_test == 'maggiore':
        # Coda di destra
        p_value = 1 - stats.norm.cdf(Z)
    elif tipo_test == 'minore':
        # Coda di sinistra
        p_value = stats.norm.cdf(Z)
    else:
        raise ValueError("tipo_test deve essere 'bilaterale', 'maggiore' o 'minore'")
        
    print(f"P-value calcolato: {p_value:.4f}")
    print(f"Livello di significatività (alfa): {alfa}")
    
    # 4. Conclusione del test
    print("-" * 30)
    if p_value < alfa:
        print("Conclusione: Rifiutiamo l'ipotesi nulla (H0).")
        print("Il risultato è statisticamente significativo al livello del 5%.")
    else:
        print("Conclusione: Non ci sono prove sufficienti per rifiutare l'ipotesi nulla (H0).")
        print("Il risultato NON è statisticamente significativo al livello del 5%.")

## Rivelazione raggi X

Alimentare il contatore GM con una tensione < 600 V.

Se si varia la tensione fra i valori riportati sulla scheda di lab si dovrebbe riscontrare, a partire dai valori più bassi, un rapido aumento dei conteggi registrati seguito da un plateau.

Con questo esperimento si vuole verificare la formazione di questa curva (cps al variare di V) e stabilire quale sia la tensione ottimale di lavoro.

**COSA DOBBIAMO MISURARE:**

Partiamo da una $V_{in} = 390 V$ e aumentiamo a passi di 2/3 V la tensione di alimentazione. Per ogni step si misurano i conteggi per un tempo di 20 secondi. Raggiunta la regione di conteggi costanti si può allargare il passo e misurare ogni 5 V.

**ANALISI DATI:** 

- Graficare i cps in funzione della tensione di alimentazione per entrambi i valori di $V_{eht}$

### Al variare della tensione

In [1]:
import numpy as np

# condizioni di lavoro
# Veht = 30  kV
# I    = 0.3 muA

cont = np.array([5,5,5,5,5])
V    = np.array([1,2,3,4,5])
errV = np.array([0.1,0.1,0.1,0.1,0.1])

freq = cont / 20  # conteggi / s

# do a plot without any fit

array([0.25, 0.25, 0.25, 0.25, 0.25])

In [ ]:
# condizioni di lavoro
# Veht = 20  kV
# I    = 0.3 muA

cont = np.array([5,5,5,5,5])
V    = np.array([1,2,3,4,5])
errV = np.array([0.1,0.1,0.1,0.1,0.1])

freq = cont / 20

# do a plot without any fit

In [1]:
VLO = ... # Il valore di tensione ottimale è ottenuto sommando al valore minimo di plateau ~ 10 V.

Si imposta HV = VLO e si procede a misurare i cps al variare della corrente.

**COSA DOBBIAMO MISURARE**

Partendo da una corrente di 0.1/0.2 $\mu A$ andiamo a prendere sempre conteggi per un tempo di 20 s aumentando la corrente a passi di 0.1 $\mu A$ fino al raggiungimento del plateau. Successivamente proseguiamo a passi di 1 $\mu A$ fino ai 20 $\mu A$. 

**ANALISI DATI:**

- Graficare i cps in funzione della corrente del filamento.

### Al variare della corrente

In [ ]:
# condizioni di lavoro
# Veht = 30  kV
# HV   = VLO

cont = np.array([5,5,5,5,5])
I    = np.array([1,2,3,4,5])
errI = np.array([0.1,0.1,0.1,0.1,0.1])

freq = cont / 20  # conteggi / s

# do a plot without any fit

## Assorbimento raggi X



Scopo di questa esperienza è studiare l’assorbimento dei raggi X per spessori crescenti di alluminio e verificare se i dati sono o meno in accordo con la legge di Lambert-Beer. $$I = I_0 e^\text{-us}$$

**COSA DOBBIAMO MISURARE**

Inseriamo un filtro di alluminio (successivamente piombo) nel sistema e andiamo a misurare i conteggi in 20 s per ogni valore di spessore dato. 

**ATTENZIONE** se dovessimo avere conteggi < 100 bisogna aumentare il tempo di acquisizione e rifare la misura.

**ANALISI DATI**

- Riportare in tabella il rateo in funzione dello spessore del filtro.

- Disegnare il grafico dei cps in funzione dello spessore e fare il fit con la relazione di Lambert-Beer.

- Disegnare il grafico di ln($\frac{I_{0}}{I}$)  in funzione dello spessore.


In [3]:
# condizioni di lavoro (ALLUMINIO)
# Veht = 30  kV
# HV   = VLO
# I    = ... (un centinaio di conteggi in 100s anche per grandi spessori di alluminio in regime lineare)

cont    = np.array([5,5,5,5,5])
spes    = np.array([0.1, 0.25, 0.35, 0.50, 0.60, 0.75, 0.85, 1.00, 1.10, 1.25, 1.35, 1.50, 1.60, 1.75, 2.00, 2.50, 3.00, 3.50]) # mm
errspes = np.ones(18) * 0.01

freq = cont / 20  # conteggi / s

# do a plot with fit [0] * TMath::Exp[[1] * x]
# do plot of log ([0] / cps) 

In [ ]:
# condizioni di lavoro (PIOMBO)
# Veht = 30  kV
# HV   = VLO
# I    = ... (un centinaio di conteggi in 100s anche per grandi spessori di alluminio in regime lineare)

cont    = np.array([5,5,5,5,5])
spes    = np.array([0.1, 0.25, 0.35, 0.50, 0.60, 0.75, 0.85, 1.00, 1.10, 1.25, 1.35, 1.50, 1.60, 1.75, 2.00, 2.50, 3.00, 3.50]) # mm
errspes = np.ones(18) * 0.01

freq = cont / 20  # conteggi / s

# do a plot without any fit

## Spettro raggi X

Questa parte dell'esperienza è dedicata allo studio dello spettro raggi X di un cristallo di NaCl.

**COSA DOBBIAMO MISURARE**

Dopo aver posizionato il cristallo si misura il numero di conteggi in 20 secondi (che strano!) muovendosi a passi di 1° nell'intervallo 11° - 35°. Nell'intorno dei picchi eseguire una scansione fine spostandosi di 1/6° ad ogni misurazione.

**ATTENZIONE** Ricordarsi che il piano di lavoro riporta i valori $2 \theta $.

**ANALISI DATI**

- Disegnare un grafico con il numero di cps in funzione dell'angolo $2 \theta $.
- Aggiornarlo con i valori di scansione fine.
- Verificare la legge di Bragg per le tre coppie di picchi caratteristici (solo per $V_{eht} = 30 kV$).
- Calcolare l'errore su $\lambda$ e su $\theta$ (usare la FWHM).
- Test Z delle lunghezze d'onda con i valori teorici

In [2]:
from math import asin, pi

# condizioni di lavoro
# Veht = 30  kV
# I    = 40  muA
# HV   = VLO

p = 0.564 # nm (p = d/2)

Kalpha = 0.1545 # nm
Kbeta  = 0.1392 # nm

thetaAlpha = [asin(n*Kalpha / p) * 180 / pi for n in range(1,4)] # fix visible orders
thetaBeta  =  [asin(n*Kbeta / p) * 180 / pi for n in range(1,4)]

thetaAlpha, thetaBeta

([15.898626890528137, 33.22116872938164, 55.266244523505215],
 [14.288736310797303, 29.57859643541395, 47.76767698362341])

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [7]:
doubleTheta = np.array([n for n in range(11,36)])
errDoubleTheta = np.ones(25)*0.1

cont = np.array([2*n for n in range(11,36)])

freq = cont / 20


# do a plot! to see the peaks

In [ ]:
# first peak (I order) 

doubleThetapeak1    = np.array([1,1,1,1,1])
errDoubleThetapeak1 = np.array([1,1,1,1,1])
cont1 = np.array([1,1,1,1,1])

freq1 = cont1 / 20

# plot! fit with (inverted) parabola

In [5]:
# second peak (I order)

doubleThetapeak2    = np.array([1,1,1,1,1])
errDoubleThetapeak2 = np.array([1,1,1,1,1])
cont2 = np.array([1,1,1,1,1])

freq2 = cont2 / 20

# plot! fit with (inverted) parabola

In [7]:
# first peak (II order) 

doubleThetapeak1    = np.array([1,1,1,1,1])
errDoubleThetapeak1 = np.array([1,1,1,1,1])
cont1 = np.array([1,1,1,1,1])

freq1 = cont2 / 20

In [ ]:
# second peak (II order)

doubleThetapeak2    = np.array([1,1,1,1,1])
errDoubleThetapeak2 = np.array([1,1,1,1,1])
cont2 = np.array([1,1,1,1,1])

freq2 = cont2 / 20

In [8]:
# first peak (III order)

doubleThetapeak1    = np.array([1,1,1,1,1])
errDoubleThetapeak1 = np.array([1,1,1,1,1])
cont1 = np.array([1,1,1,1,1])

freq1 = cont1 / 20

In [9]:
# second peak (III order)

doubleThetapeak2    = np.array([1,1,1,1,1])
errDoubleThetapeak2 = np.array([1,1,1,1,1])
cont2 = np.array([1,1,1,1,1])

freq2 = cont2 / 20

In [10]:
from math import asin, pi

# condizioni di lavoro
# Veht = 20  kV
# I    = 40  muA
# HV   = VLO

p = 0.564 # nm (p = d/2)

Kalpha = 0.1545 # nm
Kbeta  = 0.1392 # nm

thetaAlpha = [asin(n*Kalpha / p) * 180 / pi for n in range(1,4)] # fix visible orders
thetaBeta  =  [asin(n*Kbeta / p) * 180 / pi for n in range(1,4)]

thetaAlpha, thetaBeta

([15.898626890528137, 33.22116872938164, 55.266244523505215],
 [14.288736310797303, 29.57859643541395, 47.76767698362341])

In [11]:
# first peak 

doubleThetapeak1    = np.array([1,1,1,1,1])
errDoubleThetapeak1 = np.array([1,1,1,1,1])
cont1 = np.array([1,1,1,1,1])

freq1 = cont1 / 20

In [12]:
# second peak

doubleThetapeak2    = np.array([1,1,1,1,1])
errDoubleThetapeak2 = np.array([1,1,1,1,1])
cont2 = np.array([1,1,1,1,1])

freq2 = cont2 / 20

## Spettro raggi X con filtro nichel

Vogliamo valutare l'inserimento del filtro di Nickel nel cammino ottico (con una sola tensione $V_{eht}$)

**COSA DOBBIAMO MISURARE**

L'esperimento è lo stesso della terza esperienza

**ANALISI DATI**

Idem con patate :)

In [13]:
from math import asin, pi

# condizioni di lavoro
# Veht = ... (scegliamo quella che ci piace di più)
# I    = 40  muA
# HV   = VLO

p = ... # nm (p = d/2)

Kalpha = ... # nm
Kbeta  = ... # nm

thetaAlpha = [asin(n*Kalpha / p) * 180 / pi for n in range(1,4)] # fix visible orders
thetaBeta  =  [asin(n*Kbeta / p) * 180 / pi for n in range(1,4)]

thetaAlpha, thetaBeta

TypeError: unsupported operand type(s) for *: 'int' and 'ellipsis'

In [15]:
# first peak 

doubleThetapeak1    = np.array([1,1,1,1,1])
errDoubleThetapeak1 = np.array([1,1,1,1,1])
cont1 = np.array([1,1,1,1,1])

freq1 = cont1 / 20

In [14]:
# second peak

doubleThetapeak2    = np.array([1,1,1,1,1])
errDoubleThetapeak2 = np.array([1,1,1,1,1])
cont2 = np.array([1,1,1,1,1])

freq2 = cont2 / 20

## Dimensione interatomica (spettro al variare del cristallo)

Questo esperimento consente di ricavare la dimensione atomica d di alcuni cristalli conoscendo la lunghezza d’onda λ.

**COSA DOBBIAMO MISURARE**

Osservando la lucina dell’elettronica di lettura, individuare l’angolo 2θ per il primo picco di diffrazione per la radiazione Kα. Nell’intorno del picco misurare i conteggi a passi di 1/6°.

**ANALISI DATI**

- Nota λ =0.154 nm per il primo picco di diffrazione, ricavare la distanza reticolare d per i tre cristalli usando la formula di Bragg. $$ n\lambda = 2dsin(\theta)$$

- Verificare come varia la dimensione interatomica per i cristalli NaCl, KCl, RbCl in funzione del numero atomico e giustificare il risultato.

In [20]:
import numpy as np

# condizioni di lavoro
# Veht = 30  kV
# I    = 40  muA
# HV   = VLO

lambdak        = 0.154 #nm
doubleTheta    = np.array([1,1,1,1,1])
errdoubleTheta = np.array([0.1,0.1,0.1,0.1,0.1])

# Converto prima in radianti
doubleTheta_rad = np.radians(doubleTheta)

# Calcolo il valore della distanza d
d = lambdak / (2 * np.sin(doubleTheta*pi / 2))